# Harness² 101: the recursive loop, line by line

This tutorial explains the smallest complete Harness² run. It uses one task, one initial harness, and three sequential refinement rounds. Every symbol and function argument is defined before it is used.

**Environment:** Linux or WSL2, Python 3.12, repository installed with `uv pip install -e '.[render]'`. No GPU, API key, or hosted model is needed for this deterministic example.

## 0. Vocabulary and notation

| Symbol | Exact meaning |
|---|---|
| `task` | The instruction and input files given to the executor. |
| `harness` | Files controlling agent behavior: system prompt, scripts, skills, tools, memory, guardrails, subagents, and plugins. |
| `base` | The initial harness before Harness² changes it. |
| `INV` | *Invariant* proposal: a generally useful change, not tied only to this task's domain. |
| `DOM` | *Domain* proposal: a change using domain-specific knowledge, such as accounting conventions. |
| `inv1` | The invariant candidate created during round 1. |
| `dom1` | The domain candidate created during round 1. |
| `c1` | The composed harness produced at the end of round 1. |
| `k` | Number of recursive refinement rounds. Here, `k = 3`. |
| probe | Execute a candidate harness on the same task and record its behavior. |
| compose | Use candidate changes and their observed executions to produce the round's next harness. |

The notation `inv1 + dom1 -> c1` is **not arithmetic**. `+` means “the composer receives evidence from both candidates.” `->` means “produces.” Therefore:

```text
inv1 + dom1 -> c1
```

means: *probe the round-1 invariant and domain candidates; give both results to the composer; the composer outputs harness `c1`.*

## 1. Pipeline overview

For sequential recursion with `k = 3`, state flows as follows:

```text
initial harness: base
                    |
round 1: base -> propose inv1, dom1 -> probe both -> compose c1 -> probe c1
                    |                                      |
round 2:            +-- c1 -> propose inv2, dom2 -> compose c2 -> probe c2
                                                           |
round 3:                                               c2 -> propose inv3, dom3
                                                            -> compose c3 -> probe c3

final selection pool: [c1, c2, c3] -> selected harness
```

“Starts from `c1`” means the files in `c1` are copied into the round-2 proposal workspaces. Round 2 edits `c1`, not the original `base`.

## 2. Import the four objects we need

The next cell only imports classes and checks the operating system.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd().resolve()
assert (ROOT / 'pyproject.toml').is_file(), 'Start Jupyter in harness2-main.'
assert os.name != 'nt', 'Use WSL2/Linux: Harness² imports the POSIX fcntl module.'

from harness2.demo import DemoAdapter, DemoProposer
from harness2.core.schedule import run_task
from harness2.core.recursion import run_sequential


Line-by-line:

- `Path` represents file paths.
- `ROOT` is the directory where Jupyter was started; it must contain `pyproject.toml`.
- `DemoAdapter` defines the task, input files, initial harness, and execution method.
- `DemoProposer` edits harness files deterministically. In a real run, a model-backed `Proposer` performs this role.
- `run_task` orchestrates base execution, recursion, final selection, and result recording for one task.
- `run_sequential` is the lower-level function containing `for t in range(1, k + 1)`.

## 3. Construct the adapter


In [ ]:
OUTPUT = ROOT / 'runs' / 'harness2_101'
adapter = DemoAdapter(OUTPUT)
task_id = 'ledger-1'
k = 3


Each line has one job:

1. `OUTPUT` is the directory where the demo writes tasks, harness versions, executions, and summaries.
2. `DemoAdapter(OUTPUT)` creates two CSV ledger tasks and writes the initial harness under `OUTPUT/base/`.
3. `task_id` selects the first task. The string is an identifier, not the task instruction itself.
4. `k = 3` requests three recursive rounds. It does not mean three tasks.

## 4. Inspect the three inputs

A Harness² run needs: a task, an initial harness, and an executor. The adapter supplies all three.

In [ ]:
task_prompt = adapter.task_prompt(task_id)
task_directory = adapter.task_files(task_id)
base_directory = adapter.base_harness()

print('TASK PROMPT:\n', task_prompt)
print('\nTASK FILE ledger.csv:\n', (task_directory / 'ledger.csv').read_text())
print('\nBASE SYSTEM PROMPT:\n', (base_directory / 'systemprompt.md').read_text())
print('\nBASE SCRIPT:\n', (base_directory / 'scripts/summarize.py').read_text())

Variables:

- `task_prompt: str` is the instruction: sum every signed amount.
- `task_directory: Path` contains task input `ledger.csv`. Its values are `100.25`, `-20.10`, and `5.00`.
- `base_directory: Path` contains the initial harness files.
- `systemprompt.md` tells the executor which script to use.
- `scripts/summarize.py` implements the behavior. Its condition `if amount > 0` is a bug because it drops `-20.10`.

The correct result is `100.25 - 20.10 + 5.00 = 85.15`. The initial script instead returns `105.25`.

## 5. Execute the initial harness manually


In [ ]:
base_run_directory = adapter.execute(
    task_id=task_id,
    harness_dir=base_directory,
    run_id='manual/base',
)
base_answer = (base_run_directory / 'answer.txt').read_text().strip()
print(base_answer)
assert base_answer == 'Total: 105.25'


Arguments and outputs:

- `task_id=task_id` tells the adapter which prompt and CSV file to use.
- `harness_dir=base_directory` tells it which harness files control this execution.
- `run_id='manual/base'` is only a unique storage name for this execution.
- `adapter.execute(...) -> Path` runs `summarize.py ledger.csv` and returns the directory containing `answer.txt` and `trajectory.md`.
- The assertion documents the expected **wrong** base behavior. This execution evidence is what later proposal stages inspect.

## 6. Create the proposer


In [ ]:
proposer_channel = adapter.spec.channels['proposer']
proposer = DemoProposer(proposer_channel)
print(proposer_channel)


- A **channel** specifies the backend and model used for a role. The demo channel is `local/deterministic`; it makes no API call.
- `proposer` receives a copy of the current harness plus context containing the task, base trajectory, previous candidates, and previous decisions.
- In this demo, every proposal removes the substring ` if amount > 0` from `summarize.py`. In a real run, the improver model decides what files to edit.

## 7. Define the key function before calling it

The real entry point is approximately:

```python
def run_task(
    task_id: str,
    k: int,
    *,
    adapter: BenchmarkAdapter,
    prop: Proposer,
    runs_root: Path,
    modes: Sequence[str] = ('agg', 'seq'),
    ...
) -> TaskResult:
```

The `*` makes all following parameters keyword-only: we must write `adapter=adapter`. The function executes one identified task, recursively refines its harness, stores all artifacts, selects a final harness, and returns structured metadata.

### `task_id=task_id`

A keyword argument is `parameter_name=value_expression`. The left and right names have different roles:

```python
task_id = 'ledger-1'       # local variable
run_task(task_id=task_id)  # left: parameter; right: value expression
```

The right side evaluates to `'ledger-1'`. Inside `run_task`, that identifier is passed to `adapter.domain_of`, `adapter.task_prompt`, and `adapter.task_files`. It selects the task; it is not the prompt text itself.

### `k=3`

`k` is the number of task-level refinement rounds. Sequential mode executes `for t in range(1, k + 1)`, so `k=3` makes `t` equal `1`, `2`, then `3`. Each round creates `inv{t}`, `dom{t}`, and `c{t}`. Therefore this call creates nine improvement stages. `k` is not the number of tasks.

### `adapter=adapter`

The right side is our `DemoAdapter` object. An adapter connects the generic algorithm to a particular benchmark. `run_task` uses it as follows:

| Method | Information or action supplied |
|---|---|
| `base_harness()` | Locate the initial harness files. |
| `task_prompt(task_id)` | Load the instruction. |
| `task_files(task_id)` | Locate task inputs such as `ledger.csv`. |
| `execute(task_id, harness_dir, run_id)` | Execute a specific harness on the task. |
| `trajectory(run_dir, ...)` | Read actions and observations from an execution. |
| `deliverable_text(run_dir)` | Read the produced answer. |

Harness² knows the refinement algorithm; the adapter knows how this particular task is represented and executed.

### `prop=proposer`

`prop` is the function parameter; `proposer` is our `DemoProposer` object. During each round, Harness² asks it for three edits: an invariant candidate (`INV`), a domain candidate (`DOM`), and an evidence-based composition (`COMPOSE`). The proposer edits harness files but does not execute them. The adapter executes each edited harness afterward. In a real run, `Proposer.call` uses an improver model; this demo uses a deterministic edit.

### `runs_root=adapter.spec.runs_root`

This nested expression means:

```text
adapter                 DemoAdapter object
adapter.spec            benchmark configuration inside that object
adapter.spec.runs_root  Path for cached runs and harness snapshots
```

For the demo it equals `OUTPUT / 'runs'`. `run_task` writes `summary.json` and directories such as `harnesses/base`, `harnesses/inv1`, and `harnesses/c1` below it. This path is for outputs and resume state; it is not the task-input directory.

### `modes=('seq',)`

`modes` expects a sequence. `('seq',)` is a one-element tuple; the comma creates the tuple. `('seq')` would only be a string. `'seq'` selects sequential recursion, where round 2 edits `c1` and round 3 edits `c2`. The other internal mode, `'agg'`, explores candidates in parallel from the shared base. Supplying only `('seq',)` disables the parallel branch.

### Return value

`run_task(...)` returns `TaskResult`. `result.status` is the orchestration status; `result.base` describes base executions; `result.modes['seq']` contains sequential stages and finals; `result.selections['seq']` contains the chosen label and reason. Deliverable files remain on disk in execution directories.

## 8. Call the function

Every name in the following call is now defined.

In [ ]:
result = run_task(
    task_id=task_id,
    k=3,
    adapter=adapter,
    prop=proposer,
    runs_root=adapter.spec.runs_root,
    modes=('seq',),
)


Every argument:

- `task_id`: run only `ledger-1`.
- `k`: make three rounds. Each round has two proposal stages and one composition stage.
- `adapter`: provides task data, initial harness, execution, and output-reading methods.
- `prop`: edits a harness when Harness² requests an `INV`, `DOM`, or `COMPOSE` stage.
- `runs_root`: root directory for cached rollouts, harness snapshots, and `summary.json`.
- `modes=('seq',)`: a one-element tuple selecting sequential mode. The internal token `seq` means that each composed harness becomes the next round's starting state.
- `result: TaskResult`: structured metadata. It does not contain the deliverable text directly; deliverables remain in execution directories.

## 8. What `run_task` did internally

The call above performs these operations in order:

1. Read the initial harness directory into a dictionary-like harness tree.
2. Execute `k` base rollouts. These repeated references reduce dependence on a single stochastic execution; this deterministic demo produces the same result each time.
3. Call `run_sequential(...)`.
4. In each round, create and execute `inv{t}` and `dom{t}`.
5. Compose them into `c{t}`, execute `c{t}`, then set `current_harness = c{t}`.
6. Put `[c1, c2, c3]` into the final selection pool.
7. Select one final harness using task inputs and candidate deliverables.
8. Reflect on the selected change and store an experience record when evidence supports it.

In [ ]:
print('task status:', result.status)
print('base executions:', len(result.base))
print('final candidates:', result.modes['seq']['finals'])
print('selected label:', result.selections['seq']['chosen_label'])
print('selection reason:', result.selections['seq']['why'])


- `status == 'ok'` means recursion and final selection completed. It does not mean an external benchmark judge awarded a perfect score.
- `finals == ['c1', 'c2', 'c3']` lists the composed harness from each round.
- `chosen_label` identifies which final composition the selector adopted.
- `why` contains the selector's evidence-based explanation.

## 9. Print every stage in order


In [ ]:
stages = result.modes['seq']['stages']
for index, stage in enumerate(stages, start=1):
    print(
        f"{index:2}. label={stage['label']:4} "
        f"type={stage['dim']:7} "
        f"attempts={stage['attempts']} "
        f"fallback={stage['fell_back']}"
    )


Interpretation of the nine rows:

| Rows | Input harness | Proposed candidates | Composition output |
|---|---|---|---|
| 1-3 | `base` | `inv1`, `dom1` | `c1` |
| 4-6 | `c1` | `inv2`, `dom2` | `c2` |
| 7-9 | `c2` | `inv3`, `dom3` | `c3` |

`dim` is the stage type: `INV`, `DOM`, or `COMPOSE`. `attempts` counts proposal attempts. `fell_back=True` would mean a proposal remained invalid after retries, so Harness² reused the input harness instead.

## 10. Inspect each executed answer


In [ ]:
answer_files = sorted(OUTPUT.rglob('answer.txt'))
for answer_file in answer_files:
    run_name = answer_file.parent.relative_to(OUTPUT)
    answer = answer_file.read_text().strip()
    print(f'{str(run_name):75} -> {answer}')


Here, `->` means “this execution produced this answer.” Base runs should say `105.25`; fixed candidate and composition runs should say `85.15`. Harness² compares behavioral evidence—the trajectories and deliverables—not merely the text of the proposed edit.

## 11. Compare actual harness files


In [ ]:
script_files = sorted(
    adapter.spec.runs_root.rglob('harnesses/*/scripts/summarize.py')
)
for script_file in script_files:
    label = script_file.parents[1].name
    source = script_file.read_text()
    contains_bug = 'if amount > 0' in source
    print(f'{label:5} contains positive-only filter: {contains_bug}')


- `rglob(...)` recursively finds every stored version of `summarize.py`.
- `script_file.parents[1].name` extracts the harness label (`base`, `inv1`, `c1`, and so on) from its path.
- `contains_bug` checks whether that exact harness still discards negative amounts.
- A composition may be identical to a candidate. Composition means evidence-aware synthesis; it does not require combining text from both candidates.

## 12. Exact code that creates and compares `inv1`, `dom1`, and `c1`

### 12.1 Create the candidate labels

Inside `run_sequential`, the first loop iteration has `t == 1`:

```python
for t in range(1, k + 1):
    specs = [(f"inv{t}", "INV"), (f"dom{t}", "DOM")]
```

Therefore `specs` becomes `[("inv1", "INV"), ("dom1", "DOM")]`. `inv1` and `dom1` are identifiers; `INV` and `DOM` select different editing instructions.

### 12.2 Generate both candidate harnesses

The repository calls `_run_stage` once for each `(label, dimension)` pair:

```python
stages = [
    _run_stage(
        label=label,                 # 'inv1' or 'dom1'
        dim=dimension,               # 'INV' or 'DOM'
        job=prompts.propose(...),    # dimension-specific edit instruction
        fallback=current_harness,    # used if the edit is invalid
        adapter=adapter,
        prop=prop,
        task_id=task_id,
        harness_dir=current_dir,     # base harness during round 1
        pool=list(history),
        ...
    )
    for label, dimension in specs
]
```

`_run_stage` first builds `mission` (the editing instruction plus evidence) and `tree.root` (the context directory containing a writable harness copy). It then executes the operation that actually generates the candidate:

```python
reply = prop.call(
    mission,                    # what change to propose and what evidence to inspect
    tree.root,                  # context directory; edit tree.root / 'harness'
    f"{label}.a{attempt}",    # e.g. 'inv1.a1' or 'dom1.a1'
    harvest="context/harness", # return the edited harness directory
    max_attempts=TRANSPORT_ATTEMPTS,
)
```

The **label does not control the edit**. `inv1` and `dom1` are tracking names. The `dimension` value controls the edit by changing the job prompt before `prop.call` runs:

```python
label, dimension = specs[0]  # label='inv1', dimension='INV'
job = prompts.propose(benchmark_prompts, dimension, surface, ...)
mission = context.mission(tree, job=job)
reply = prop.call(mission, tree.root, f"{label}.a{attempt}", ...)
```

The same code runs again with `label='dom1'` and `dimension='DOM'`. Inside `prompts.propose`, the dimension selects a different prompt block:

```python
block = st.inv if dimension == 'INV' else st.dom
```

Therefore `prop.call()` edits the copied harness files differently:

- With `dimension='INV'`, `mission` asks for an invariant/general improvement to **how the agent works**, intended to transfer across domains.
- With `dimension='DOM'`, `mission` asks for a domain-specific improvement to **what a practitioner knows**, using knowledge relevant to the current domain.

`prop.call` does not receive the word `INV` or `DOM` as a separate parameter. It receives the already-generated `mission` text. The label appears in `f"{label}.a{attempt}"` only to name and audit the proposer session. `reply.workspace` is the path to the files that the proposer edited.

The edited directory becomes a candidate only after `_run_stage` reads and validates it:

```python
new_tree = harness_tree.read_tree(surface, reply.workspace)
kept_base, error = _accept_tree(surface, adapter, new_tree, fallback, label)
return Stage(label=label, dim=dim, tree=new_tree, ...)
```

Therefore `specs` defines two jobs; `_run_stage` prepares each job; `prop.call` edits the harness; and `read_tree` converts the edited files into the actual `inv1` or `dom1` candidate tree. The candidates have not yet been executed—that happens in the next probe step.

In [ ]:
# Build the two real job prompts and display only their differing lines.
from difflib import unified_diff
from harness2.core import prompts as prompt_builder
from harness2.core.spec import resolve_prompts

benchmark_prompts = resolve_prompts(adapter, task_id)
surface = adapter.substrate.surface()

inv_job = prompt_builder.propose(
    benchmark_prompts, 'INV', surface, k=3, index=1, round_t=1
)
dom_job = prompt_builder.propose(
    benchmark_prompts, 'DOM', surface, k=3, index=1, round_t=1
)

difference = unified_diff(
    inv_job.splitlines(),
    dom_job.splitlines(),
    fromfile='INV job sent through mission to prop.call',
    tofile='DOM job sent through mission to prop.call',
    lineterm='',
)
print('\n'.join(difference))


This cell proves the distinction directly: the same prompt-building function receives a different `dimension`, producing different job text. `_run_stage` embeds that job in `mission`; then `prop.call(mission, ...)` acts on those different instructions. Neither `inv1` nor `dom1` itself changes behavior—the label only connects the prompt, edited harness, rollout, and notes under one name.

### 12.3 Probe both candidates

The following code executes both candidate trees:

```python
result.rollouts.update(
    _roll_many(
        adapter, ledger, task_id,
        [(stage.label, stage.tree) for stage in stages],
        run_id_of, work_dir, width=rollout_workers,
    )
)
```

The list passed to `_roll_many` is approximately `[("inv1", inv1_tree), ("dom1", dom1_tree)]`. For each item, `_roll_many` writes the harness files and calls `adapter.execute`. The resulting `RunView` records the execution directory, success state, trajectory, and deliverable. These observed executions are the probe evidence.

### 12.4 Construct the comparison pool

Harness² pairs each candidate's files with its observed execution:

```python
round_pool = list(history) + [
    (
        stage.label,
        work_dir / "harnesses" / stage.label,
        result.rollouts.get(stage.label),
    )
    for stage in stages
]
```

In round 1, `history` is empty. The pool therefore contains `('inv1', inv1_directory, inv1_RunView)` and `('dom1', dom1_directory, dom1_RunView)`. The composer can inspect both **what changed** and **what happened after execution**. There is no numeric scoring function at this point; the comparison is an evidence-based model editing step.

### 12.5 Generate and execute `c1`

The first composition label is created and passed to `_run_stage`:

```python
ck = f"c{t}"  # t == 1, so ck == 'c1'
comp = _run_stage(
    label=ck,
    dim="COMPOSE",
    job=prompts.compose_round(...),
    fallback=current_harness,
    harness_dir=current_dir,
    pool=round_pool,       # inv1/dom1 files and probe results
    decisions={...},       # proposal notes
    adapter=adapter, prop=prop, task_id=task_id, ...
)
```

`context.build` copies the pool evidence into the composer's context. `prop.call` reads candidate diffs, trajectories, and deliverables, then edits a fresh copy of the current harness. That edited tree is `c1`. Harness² executes it and carries it forward:

```python
result.rollouts['c1'] = _roll(..., 'c1', comp.tree, ...)
history = round_pool + [('c1', c1_directory, result.rollouts['c1'])]
current_harness = comp.tree  # round 2 now starts from c1
```

### 12.6 Composition is different from final selection

There are two comparisons:

1. **Within each round**, composition compares `inv1` and `dom1` evidence to generate `c1`. It may combine edits, retain one edit, or keep the input harness.
2. **After all rounds**, `_run_one_mode` builds a pool containing final compositions `[c1, c2, c3]` and calls `selection.run_selection(...)`. The selector opens every candidate deliverable and chooses one adopted harness. Labels are anonymized during selection so names such as `c3` do not bias the choice.

In [ ]:
# Inspect the real round-1 stage records created earlier.
round_1 = [
    stage for stage in result.modes['seq']['stages']
    if stage['label'] in {'inv1', 'dom1', 'c1'}
]
for stage in round_1:
    print(
        f"{stage['label']}: type={stage['dim']}, "
        f"notes={stage['notes']}"
    )


## 13. The complete recursive loop

The following is a faithful, shortened reading of `run_sequential`. The comments explain every state change.

In [ ]:
current_harness = 'base'          # conceptual label; real code stores a file tree
history = []                      # earlier candidates, compositions, and executions

for t in range(1, k + 1):         # t takes values 1, 2, 3
    inv_label = f'inv{t}'         # inv1, then inv2, then inv3
    dom_label = f'dom{t}'         # dom1, then dom2, then dom3
    composed_label = f'c{t}'      # c1, then c2, then c3

    print(f'round {t}: input={current_harness}')
    print(f'  propose and probe {inv_label} and {dom_label}')
    print(f'  compose evidence into {composed_label}')

    history.extend([inv_label, dom_label, composed_label])
    current_harness = composed_label  # next round edits this composition

print('final current_harness:', current_harness)
print('history:', history)


This cell illustrates control flow only; the earlier `run_task(...)` cell performed the real proposals and executions. In the repository implementation, `current_harness` is a dictionary mapping relative filenames to file contents, and `history` contains `(label, harness_directory, RunView)` tuples rather than strings.

In [ ]:
import inspect
print(inspect.getsource(run_sequential))


## 14. The second recursion level: across tasks

Task-level recursion improves a harness for the current task. Harnessing-level recursion carries lessons across tasks:

```python
library = []
for task_id in task_stream:
    selected_harness, evidence = run_task(task_id, library)
    library.append((selected_harness, evidence))
```

The actual function is `run_domain()`. Before task 2, the library contains the adopted harness and evidence from task 1. The proposer can use that experience, but each task still begins from the benchmark's base harness; the library guides proposals rather than silently replacing the base.

## 15. Mapping the demo to a real run

| Demo component | Real benchmark component |
|---|---|
| CSV ledger task | JobBench, LAB, or WorkBuddy task |
| `DemoAdapter` | Benchmark-specific adapter |
| deterministic Python executor | OpenCode or Codex executor backed by a hosted model |
| `DemoProposer` string replacement | Improver model editing harness files |
| deterministic selector | Selector model comparing final deliverables |
| `answer.txt` | Benchmark deliverable files |

A minimal real run is:

```bash
harness2-setup jb
harness2-check --bench jb
harness2 --bench jb --k 1 --mode sequential --max-tasks 1 --dry-run
harness2 --bench jb --k 1 --mode sequential --max-tasks 1
```

The last command incurs model API costs. Start with `k=1` and one task.